In [ ]:
# --- Repository setup ---
import os
from pathlib import Path

# Find the repository root (the folder that contains data/graphs) and work from there,
# so every relative path below resolves no matter where Jupyter was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'graphs').is_dir())
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)


In [1]:
# Optional (recommended): install dependencies into the current Jupyter kernel env
%pip install -q neo4j openai


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# No-op cell (kept for notebook structure)


In [2]:
from __future__ import annotations

import os
from openai import OpenAI

# Uses OPENAI_API_KEY from your local environment.
# In your shell:
#   export OPENAI_API_KEY='...'
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in your environment before running model calls')
client = OpenAI()

OPENAI_YESNO_MODEL = os.getenv('OPENAI_YESNO_MODEL', 'gpt-4.1-mini')


In [3]:
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

from neo4j import GraphDatabase


In [4]:
class Neo4jConnector:

    def __init__(self, uri: str, username: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(username, password))

    def close(self):
        self.driver.close()

    def run_query(self, query: str, parameters: dict | None = None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [r.data() for r in result]


In [5]:
class GraphRepository:

    def __init__(self, db: Neo4jConnector):
        self.db = db

    def get_start_node(self, module_id: str):
        # Finds node like DEP_A1 when module_id is "DEP"
        query = """
        MATCH (s:Step)
        WHERE s.id STARTS WITH $module_prefix AND s.is_starting = true
        RETURN s.id AS id, s.text AS text, s.type AS type, s.is_starting AS is_starting, s.target_module AS target_module
        ORDER BY s.id
        LIMIT 1
        """
        result = self.db.run_query(query, {"module_prefix": f"{module_id}_"})
        return result[0] if result else None

    def get_next_node(self, node_id: str, condition: str):
        query = """
        MATCH (a:Step {id: $id})-[r]->(b:Step)
        WHERE r.condition = $condition
        RETURN b.id AS id, b.text AS text, b.type AS type, b.is_starting AS is_starting, b.target_module AS target_module
        ORDER BY b.id
        LIMIT 1
        """
        result = self.db.run_query(query, {"id": node_id, "condition": condition})
        return result[0] if result else None

    def get_fallback_exit_node(self, node_id: str):
        query = """
        MATCH (a:Step {id: $id})-[r]->(b:Step {type: 'exit'})
        RETURN b.id AS id, b.text AS text, b.type AS type, b.is_starting AS is_starting
        ORDER BY b.id
        LIMIT 1
        """
        result = self.db.run_query(query, {"id": node_id})
        return result[0] if result else None


StopReason = Literal[
    'no_answer',
    'dead_end_no_edge',
    'exit_reached',
    'missing_target_module_graph',
    'max_referral_hops',
    'max_iterations',
    'unknown_node_type',
]


@dataclass
class RunResult:
    module_id: str
    collected_answers: dict[str, str]
    step_logs: list[dict[str, Any]]
    transitions: list[dict[str, Any]]
    stop_reason: StopReason
    stop_node_id: str
    usage: dict[str, Any]


class AssessmentEngine:

    def __init__(self, graph_repo: GraphRepository):
        self.graph_repo = graph_repo

    def run(
        self,
        start_module_id: str,
        responder,
        *,
        max_iter: int = 100,
        max_referral_hops: int = 1,
        verbose: bool = True,
    ) -> RunResult:
        collected: dict[str, str] = {}
        step_logs: list[dict[str, Any]] = []
        transitions: list[dict[str, Any]] = []
        referral_hops = 0
        usage = {
            'vp': {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0},
        }

        node = self.graph_repo.get_start_node(start_module_id)
        if not node:
            raise RuntimeError(f"Could not find starting node for module {start_module_id}")

        for _ in range(max_iter):
            node_id = str(node.get('id'))
            text = str(node.get('text', '') or '')
            node_type = str(node.get('type') or '')
            is_starting = bool(node.get('is_starting', False))
            target_module = node.get('target_module')

            log_entry: dict[str, Any] = {
                'id': node_id,
                'type': node_type,
                'is_starting': is_starting,
                'text': text,
            }

            if node_type == 'assessment':
                ans, u = responder.answer(text)
                ans = str(ans).strip().upper()
                if ans not in ('YES', 'NO', 'NO_ANSWER'):
                    ans = 'NO_ANSWER'

                usage['vp']['calls'] += 1
                usage['vp']['input_tokens'] += int(u.get('input_tokens', 0) or 0)
                usage['vp']['output_tokens'] += int(u.get('output_tokens', 0) or 0)
                usage['vp']['total_tokens'] += int(u.get('total_tokens', 0) or 0)

                collected[node_id] = ans
                log_entry['user_answer'] = ans
                step_logs.append(log_entry)
                if verbose:
                    print(json.dumps(log_entry, ensure_ascii=False))

                if ans == 'NO_ANSWER':
                    return RunResult(start_module_id, collected, step_logs, transitions, 'no_answer', node_id, usage)

                nxt = self.graph_repo.get_next_node(node_id, ans)
                if not nxt:
                    return RunResult(start_module_id, collected, step_logs, transitions, 'dead_end_no_edge', node_id, usage)
                transitions.append({'from': node_id, 'answer': ans, 'to': str(nxt.get('id'))})
                node = nxt

            elif node_type in ('management', 'diagnosis'):
                step_logs.append(log_entry)
                if verbose:
                    print(json.dumps(log_entry, ensure_ascii=False))

                nxt = self.graph_repo.get_next_node(node_id, 'NEXT')
                if not nxt:
                    return RunResult(start_module_id, collected, step_logs, transitions, 'dead_end_no_edge', node_id, usage)
                transitions.append({'from': node_id, 'answer': 'NEXT', 'to': str(nxt.get('id'))})
                node = nxt

            elif node_type == 'referral':
                log_entry['target_module'] = target_module
                step_logs.append(log_entry)
                if verbose:
                    print(json.dumps(log_entry, ensure_ascii=False))

                if referral_hops >= max_referral_hops:
                    return RunResult(start_module_id, collected, step_logs, transitions, 'max_referral_hops', node_id, usage)

                target_module_id = str(target_module or '')
                if not target_module_id:
                    return RunResult(start_module_id, collected, step_logs, transitions, 'missing_target_module_graph', node_id, usage)

                next_node = self.graph_repo.get_start_node(target_module_id)
                if next_node:
                    referral_hops += 1
                    transitions.append({'from': node_id, 'answer': 'REFERRAL', 'to': str(next_node.get('id')), 'target_module': target_module_id})
                    node = next_node
                else:
                    # If your Neo4j instance doesn't have the referred module loaded, attempt a safe exit.
                    exit_node = self.graph_repo.get_fallback_exit_node(node_id)
                    if exit_node:
                        transitions.append({'from': node_id, 'answer': 'FALLBACK_EXIT', 'to': str(exit_node.get('id'))})
                        node = exit_node
                    else:
                        return RunResult(start_module_id, collected, step_logs, transitions, 'missing_target_module_graph', node_id, usage)

            elif node_type == 'exit':
                step_logs.append(log_entry)
                if verbose:
                    print(json.dumps(log_entry, ensure_ascii=False))
                return RunResult(start_module_id, collected, step_logs, transitions, 'exit_reached', node_id, usage)

            else:
                step_logs.append(log_entry)
                if verbose:
                    print(json.dumps(log_entry, ensure_ascii=False))
                return RunResult(start_module_id, collected, step_logs, transitions, 'unknown_node_type', node_id, usage)

        # Safety stop
        return RunResult(start_module_id, collected, step_logs, transitions, 'max_iterations', str(node.get('id')), usage)


In [6]:
# Neo4j connection (set these in your local environment)
# Example:
#   export NEO4J_URI='bolt://localhost:7687'
#   export NEO4J_USERNAME='neo4j'
#   export NEO4J_PASSWORD='...'

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD') 




if not (NEO4J_URI and NEO4J_USERNAME and NEO4J_PASSWORD):
    raise RuntimeError('Set NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD in your environment')

db = Neo4jConnector(uri=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
repo = GraphRepository(db)
engine = AssessmentEngine(repo)


In [7]:
@dataclass
class StoryCase:
    case_id: str
    source_module: str
    user_story: str
    expected_node_sequence: list[dict[str, Any]]
    expected_diagnosis_node_id: str | None
    expected_exit_node_id: str | None


class LLMPatientResponder:
    # Answers ONLY YES/NO/NO_ANSWER based on the user_story text.

    def __init__(self, *, user_story: str, model: str = OPENAI_YESNO_MODEL):
        if not user_story or not isinstance(user_story, str):
            raise ValueError('user_story must be a non-empty string')
        self.user_story = user_story
        self.model = model

    def answer(self, question: str) -> tuple[str, dict[str, int]]:
        prompt = (
            "You are simulating a virtual patient during a clinical assessment.\n"
            "Answer the clinician's question using ONLY the patient story below.\n\n"
            "PATIENT STORY:\n"
            f"{self.user_story}\n\n"
            "CLINICIAN QUESTION:\n"
            f"{question}\n\n"
            "RULES:\n"
            "- Return exactly one of: YES, NO, NO_ANSWER.\n"
            "- Use NO_ANSWER only if the story truly does not contain enough info.\n"
        )

        resp = client.responses.create(
            model=self.model,
            input=[
                {
                    'role': 'developer',
                    'content': 'Output must be one token: YES, NO, or NO_ANSWER.',
                },
                {'role': 'user', 'content': prompt},
            ],
            temperature=0,
            max_output_tokens=25,
        )

        text = (resp.output_text or '').strip().upper()
        if text.startswith('YES'):
            ans = 'YES'
        elif text.startswith('NO_ANSWER') or text == 'NO ANSWER':
            ans = 'NO_ANSWER'
        elif text.startswith('NO'):
            ans = 'NO'
        else:
            ans = 'NO_ANSWER'

        u = getattr(resp, 'usage', None)
        usage = {
            'input_tokens': int(getattr(u, 'input_tokens', 0) or 0),
            'output_tokens': int(getattr(u, 'output_tokens', 0) or 0),
            'total_tokens': int(getattr(u, 'total_tokens', 0) or 0),
        }
        return ans, usage


In [8]:
def extract_engine_outcomes(step_logs: list[dict[str, Any]]):
    return {
        'engine_diagnosis_node_id': next((s.get('id') for s in step_logs if s.get('type') == 'diagnosis'), None),
        'engine_management_node_ids': [s.get('id') for s in step_logs if s.get('type') == 'management'],
        'engine_referrals': [s.get('target_module') for s in step_logs if s.get('type') == 'referral'],
        'engine_exit_node_id': next((s.get('id') for s in step_logs if s.get('type') == 'exit'), None),
    }


In [9]:
def _expected_assessment_answers(case: StoryCase) -> dict[str, str]:
    # Map assessment node_id -> expected YES/NO (ignoring NEXT)
    out: dict[str, str] = {}
    for node in case.expected_node_sequence:
        if not isinstance(node, dict):
            continue
        if node.get('type') != 'assessment':
            continue
        cond = node.get('condition_met')
        if cond in ('YES', 'NO'):
            out[str(node.get('id'))] = cond
    return out


def evaluate_engine_run(case: StoryCase, run: RunResult) -> dict[str, Any]:
    expected_node_ids = [n.get('id') for n in case.expected_node_sequence if isinstance(n, dict)]
    actual_node_ids = [s.get('id') for s in run.step_logs]

    # Baseline-style finding comparison: expected assessment condition_met vs actual user_answer
    expected_findings = _expected_assessment_answers(case)
    actual_findings = {
        s.get('id'): s.get('user_answer')
        for s in run.step_logs
        if s.get('type') == 'assessment' and s.get('user_answer') in ('YES', 'NO')
    }

    matched = 0
    missing = []
    mismatched = []
    for nid, exp in expected_findings.items():
        got = actual_findings.get(nid)
        if got is None:
            missing.append({'node_id': nid, 'expected': exp, 'actual': None})
            continue
        if got == exp:
            matched += 1
        else:
            mismatched.append({'node_id': nid, 'expected': exp, 'actual': got})

    total = len(expected_findings)
    finding_accuracy = (matched / total) if total else None

    extra = [
        {'node_id': nid, 'expected': None, 'actual': ans}
        for nid, ans in actual_findings.items()
        if nid not in expected_findings
    ]

    outcomes = extract_engine_outcomes(run.step_logs)

    diagnosis_ok = True
    if case.expected_diagnosis_node_id:
        diagnosis_ok = outcomes.get('engine_diagnosis_node_id') == case.expected_diagnosis_node_id

    exit_ok = True
    if case.expected_exit_node_id:
        exit_ok = outcomes.get('engine_exit_node_id') == case.expected_exit_node_id

    prefix_ok = actual_node_ids == expected_node_ids[: len(actual_node_ids)]
    exact_ok = actual_node_ids == expected_node_ids

    vp_tokens = int(run.usage.get('vp', {}).get('total_tokens', 0) or 0)

    return {
        'case_id': case.case_id,
        'prefix_ok': prefix_ok,
        'exact_ok': exact_ok,
        'finding_accuracy': finding_accuracy,
        'missing_findings': missing,
        'mismatched_findings': mismatched,
        'extra_findings': extra,
        'counts': {
            'expected_assessment_findings': total,
            'matched': matched,
            'missing': len(missing),
            'mismatched': len(mismatched),
            'extra': len(extra),
        },
        # Backwards compatible alias
        'mismatched_answers': mismatched,
        'diagnosis_ok': diagnosis_ok,
        'exit_ok': exit_ok,
        'actual_node_ids': actual_node_ids,
        'expected_node_ids': expected_node_ids,
        'engine_outcomes': outcomes,
        'stop_reason': run.stop_reason,
        'stop_node_id': run.stop_node_id,
        'vp_total_tokens': vp_tokens,
    }


In [ ]:
def load_cases_from_dataset(path: str | Path) -> list[dict[str, Any]]:
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    cases = data.get('cases', [])
    if not isinstance(cases, list):
        raise ValueError('Invalid dataset: cases must be a list')
    return cases


def to_story_case(case_dict: dict[str, Any]) -> StoryCase:
    return StoryCase(
        case_id=str(case_dict.get('case_id')),
        source_module=str(case_dict.get('source_module')),
        user_story=str(case_dict.get('user_story', '') or ''),
        expected_node_sequence=case_dict.get('node_sequence', []) or [],
        expected_diagnosis_node_id=case_dict.get('diagnosis_node_id'),
        expected_exit_node_id=case_dict.get('exit_node_id'),
    )


def run_dataset_file(
    dataset_path: str | Path,
    *,
    limit: int | None = None,
    verbose: bool = True,
    save_dir: Path | None = None,
) -> list[dict[str, Any]]:
    dataset_path = Path(dataset_path)
    cases = load_cases_from_dataset(dataset_path)
    results: list[dict[str, Any]] = []

    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)

    for idx, case_dict in enumerate(cases):
        if limit is not None and idx >= limit:
            break
        case = to_story_case(case_dict)
        responder = LLMPatientResponder(user_story=case.user_story)
        run = engine.run(case.source_module, responder, max_referral_hops=1, verbose=verbose)
        eval_result = evaluate_engine_run(case, run)
        results.append(eval_result)

        if save_dir is not None:
            stem = dataset_path.stem
            cid = str(case.case_id or f'case{idx}')
            out_path = save_dir / f'{stem}__{cid}__graph.json'
            payload = {
                **run.__dict__,
                'dataset_path': str(dataset_path),
                'case_id': cid,
                'expected_node_sequence': case.expected_node_sequence,
                'evaluation': eval_result,
                'vp_total_tokens': int(run.usage.get('vp', {}).get('total_tokens', 0) or 0),
            }
            out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')

    return results


def run_all_datasets_in_dir(dir_path: str | Path, *, glob_pattern: str = '*.json', limit_per_file: int | None = None) -> dict[str, list[dict[str, Any]]]:
    dir_path = Path(dir_path)
    out: dict[str, list[dict[str, Any]]] = {}
    for p in sorted(dir_path.rglob(glob_pattern)):
        out[str(p)] = run_dataset_file(p, limit=limit_per_file)
    return out


def print_summary(label: str, results: list[dict[str, Any]]):
    total = len(results)
    exact = sum(
        1
        for r in results
        if r.get('exact_ok')
        and not r.get('mismatched_answers')
        and r.get('diagnosis_ok')
        and r.get('exit_ok')
    )
    prefix = sum(1 for r in results if r.get('prefix_ok'))

    vp_tokens = [int(r.get('vp_total_tokens', 0) or 0) for r in results]
    vp_total = sum(vp_tokens)

    print(f'\n=== {label} ===')
    print(f'Total cases: {total}')
    print(f'Exact path match + answers + dx/exit: {exact}/{total}')
    print(f'Prefix path match: {prefix}/{total}')
    print(f'VP total tokens (sum): {vp_total}')
    if total:
        print(f'VP total tokens (avg): {vp_total / total:.1f}')

    bad = [
        r
        for r in results
        if r.get('mismatched_answers')
        or not r.get('exit_ok')
        or not r.get('diagnosis_ok')
        or r.get('stop_reason') not in (None, 'exit_reached')
    ]
    if bad:
        print('First few issues:')
        for r in bad[:5]:
            mm = r.get('mismatched_answers', [])
            print('-', r['case_id'], 'stop:', r.get('stop_reason'), 'mismatches:', len(mm), 'dx_ok:', r.get('diagnosis_ok'), 'exit_ok:', r.get('exit_ok'))


In [13]:
# Run ONE module dataset JSON file (one at a time)

EVAL_OUTPUT_ROOT = Path('results/raw_runs/S1_symbolic')
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path('data/cases/virtual_patients/DEP/DEP_single_hop_single_hop_positive_1.json')

results = run_dataset_file(DATASET_PATH, limit=None, verbose=True, save_dir=EVAL_OUTPUT_ROOT)
print_summary(str(DATASET_PATH), results)

# Save the aggregated evaluation results for this dataset
out_path = EVAL_OUTPUT_ROOT / f'{DATASET_PATH.stem}__graph_eval.json'
out_path.write_text(json.dumps({'dataset_path': str(DATASET_PATH), 'results': results}, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', out_path)

# To run EVERY dataset JSON under data/cases/virtual_patients:
# all_results = run_all_datasets_in_dir(Path('data/cases/virtual_patients'), limit_per_file=1)
# for path, res in all_results.items():
#     print_summary(path, res)


{"id": "DEP_A1", "type": "assessment", "is_starting": true, "text": "Has the person had at least one of the following core symptoms of depression for at least 2 weeks? (Persistent depressed mood, markedly diminished interest in or pleasure from activities)", "user_answer": "YES"}
{"id": "DEP_A2", "type": "assessment", "is_starting": false, "text": "Has the person had several of the following additional symptoms for at least 2 weeks: Disturbed sleep, indecisiveness, change in appetite or weight, agitation, beliefs of worthlessness, slow moving/talking, fatigue, hopelessness, reduced concentration, suicidal thoughts or acts?", "user_answer": "YES"}
{"id": "DEP_A3", "type": "assessment", "is_starting": false, "text": "Does the person have considerable difficulty with daily functioning in personal, family, social, educational, occupational or other areas?", "user_answer": "YES"}
{"id": "DEP_A4", "type": "assessment", "is_starting": false, "text": "Are there other possible explanations for 

In [24]:

EVAL_OUTPUT_ROOT = Path('results/raw_runs/S1_symbolic')
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path('data/cases/virtual_patients/DEP/DEP_single_hop_single_hop_negative_1.json')

results = run_dataset_file(DATASET_PATH, limit=None, verbose=True, save_dir=EVAL_OUTPUT_ROOT)
print_summary(str(DATASET_PATH), results)

# Save the aggregated evaluation results for this dataset
out_path = EVAL_OUTPUT_ROOT / f'{DATASET_PATH.stem}__graph_eval.json'
out_path.write_text(json.dumps({'dataset_path': str(DATASET_PATH), 'results': results}, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', out_path)

# To run EVERY dataset JSON under data/cases/virtual_patients:
# all_results = run_all_datasets_in_dir(Path('data/cases/virtual_patients'), limit_per_file=1)
# for path, res in all_results.items():
#     print_summary(path, res)

{"id": "DEP_A1", "type": "assessment", "is_starting": true, "text": "Has the person had at least one of the following core symptoms of depression for at least 2 weeks? (Persistent depressed mood, markedly diminished interest in or pleasure from activities)", "user_answer": "YES"}
{"id": "DEP_A2", "type": "assessment", "is_starting": false, "text": "Has the person had several of the following additional symptoms for at least 2 weeks: Disturbed sleep, indecisiveness, change in appetite or weight, agitation, beliefs of worthlessness, slow moving/talking, fatigue, hopelessness, reduced concentration, suicidal thoughts or acts?", "user_answer": "YES"}
{"id": "DEP_A3", "type": "assessment", "is_starting": false, "text": "Does the person have considerable difficulty with daily functioning in personal, family, social, educational, occupational or other areas?", "user_answer": "YES"}
{"id": "DEP_A4", "type": "assessment", "is_starting": false, "text": "Are there other possible explanations for 

In [14]:
EVAL_OUTPUT_ROOT = Path('results/raw_runs/S1_symbolic')
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path('data/cases/virtual_patients/DEP/DEP_to_SUI_multi_hop_multi_hop_positive_1.json')

results = run_dataset_file(DATASET_PATH, limit=None, verbose=True, save_dir=EVAL_OUTPUT_ROOT)
print_summary(str(DATASET_PATH), results)

# Save the aggregated evaluation results for this dataset
out_path = EVAL_OUTPUT_ROOT / f'{DATASET_PATH.stem}__graph_eval.json'
out_path.write_text(json.dumps({'dataset_path': str(DATASET_PATH), 'results': results}, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', out_path)

{"id": "DEP_A1", "type": "assessment", "is_starting": true, "text": "Has the person had at least one of the following core symptoms of depression for at least 2 weeks? (Persistent depressed mood, markedly diminished interest in or pleasure from activities)", "user_answer": "YES"}
{"id": "DEP_A2", "type": "assessment", "is_starting": false, "text": "Has the person had several of the following additional symptoms for at least 2 weeks: Disturbed sleep, indecisiveness, change in appetite or weight, agitation, beliefs of worthlessness, slow moving/talking, fatigue, hopelessness, reduced concentration, suicidal thoughts or acts?", "user_answer": "YES"}
{"id": "DEP_A3", "type": "assessment", "is_starting": false, "text": "Does the person have considerable difficulty with daily functioning in personal, family, social, educational, occupational or other areas?", "user_answer": "YES"}
{"id": "DEP_A4", "type": "assessment", "is_starting": false, "text": "Are there other possible explanations for 

In [23]:
EVAL_OUTPUT_ROOT = Path('results/raw_runs/S1_symbolic')
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path('data/cases/virtual_patients/DEP/DEP_to_CMH_multi_hop_multi_hop_positive_1.json')

results = run_dataset_file(DATASET_PATH, limit=None, verbose=True, save_dir=EVAL_OUTPUT_ROOT)
print_summary(str(DATASET_PATH), results)

# Save the aggregated evaluation results for this dataset
out_path = EVAL_OUTPUT_ROOT / f'{DATASET_PATH.stem}__graph_eval.json'
out_path.write_text(json.dumps({'dataset_path': str(DATASET_PATH), 'results': results}, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', out_path)

{"id": "DEP_A1", "type": "assessment", "is_starting": true, "text": "Has the person had at least one of the following core symptoms of depression for at least 2 weeks? (Persistent depressed mood, markedly diminished interest in or pleasure from activities)", "user_answer": "YES"}
{"id": "DEP_A2", "type": "assessment", "is_starting": false, "text": "Has the person had several of the following additional symptoms for at least 2 weeks: Disturbed sleep, indecisiveness, change in appetite or weight, agitation, beliefs of worthlessness, slow moving/talking, fatigue, hopelessness, reduced concentration, suicidal thoughts or acts?", "user_answer": "YES"}
{"id": "DEP_A3", "type": "assessment", "is_starting": false, "text": "Does the person have considerable difficulty with daily functioning in personal, family, social, educational, occupational or other areas?", "user_answer": "YES"}
{"id": "DEP_A4", "type": "assessment", "is_starting": false, "text": "Are there other possible explanations for 

In [25]:
EVAL_OUTPUT_ROOT = Path('results/raw_runs/S1_symbolic')
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path('data/cases/virtual_patients/DEP/DEP_to_PSY_multi_hop_multi_hop_positive_1.json')

results = run_dataset_file(DATASET_PATH, limit=None, verbose=True, save_dir=EVAL_OUTPUT_ROOT)
print_summary(str(DATASET_PATH), results)

# Save the aggregated evaluation results for this dataset
out_path = EVAL_OUTPUT_ROOT / f'{DATASET_PATH.stem}__graph_eval.json'
out_path.write_text(json.dumps({'dataset_path': str(DATASET_PATH), 'results': results}, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', out_path)

{"id": "DEP_A1", "type": "assessment", "is_starting": true, "text": "Has the person had at least one of the following core symptoms of depression for at least 2 weeks? (Persistent depressed mood, markedly diminished interest in or pleasure from activities)", "user_answer": "YES"}
{"id": "DEP_A2", "type": "assessment", "is_starting": false, "text": "Has the person had several of the following additional symptoms for at least 2 weeks: Disturbed sleep, indecisiveness, change in appetite or weight, agitation, beliefs of worthlessness, slow moving/talking, fatigue, hopelessness, reduced concentration, suicidal thoughts or acts?", "user_answer": "YES"}
{"id": "DEP_A3", "type": "assessment", "is_starting": false, "text": "Does the person have considerable difficulty with daily functioning in personal, family, social, educational, occupational or other areas?", "user_answer": "YES"}
{"id": "DEP_A4", "type": "assessment", "is_starting": false, "text": "Are there other possible explanations for 

In [26]:
EVAL_OUTPUT_ROOT = Path('results/raw_runs/S1_symbolic')
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path('data/cases/virtual_patients/DEP/DEP_to_OTH_multi_hop_multi_hop_positive_1.json')

results = run_dataset_file(DATASET_PATH, limit=None, verbose=True, save_dir=EVAL_OUTPUT_ROOT)
print_summary(str(DATASET_PATH), results)

# Save the aggregated evaluation results for this dataset
out_path = EVAL_OUTPUT_ROOT / f'{DATASET_PATH.stem}__graph_eval.json'
out_path.write_text(json.dumps({'dataset_path': str(DATASET_PATH), 'results': results}, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', out_path)

{"id": "DEP_A1", "type": "assessment", "is_starting": true, "text": "Has the person had at least one of the following core symptoms of depression for at least 2 weeks? (Persistent depressed mood, markedly diminished interest in or pleasure from activities)", "user_answer": "NO"}
{"id": "DEP_R1", "type": "referral", "is_starting": false, "text": "Depression is unlikely. Go to Other Significant Mental Health Complaints.", "target_module": "OTH"}
{"id": "OTH_A1", "type": "assessment", "is_starting": true, "text": "Is there a physical cause that fully explains the presenting symptoms?", "user_answer": "NO"}
{"id": "OTH_A2", "type": "assessment", "is_starting": false, "text": "Is this depression or another MNS condition discussed in another module of this guide?", "user_answer": "NO"}
{"id": "OTH_A3", "type": "assessment", "is_starting": false, "text": "Is the person seeking help to relieve symptoms or having considerable difficulty with daily functioning because of their symptoms?", "user_

In [12]:
EVAL_OUTPUT_ROOT = Path('results/raw_runs/S1_symbolic')
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [17]:
from pathlib import Path
import json

DATASET_DIR = Path("data/cases/virtual_patients/SUI")

# Create output subfolder: EVAL_OUTPUT_ROOT/DEP
output_dir = EVAL_OUTPUT_ROOT / "SUI"
output_dir.mkdir(parents=True, exist_ok=True)

for dataset_path in sorted(DATASET_DIR.glob("*.json")):
    print(f"\nProcessing: {dataset_path.name}")

    results = run_dataset_file(
        dataset_path,
        limit=None,
        verbose=True,
        save_dir=output_dir,
    )

    print_summary(str(dataset_path), results)

    out_path = output_dir / f"{dataset_path.stem}__graph_eval.json"

    out_path.write_text(
        json.dumps(
            {
                "dataset_path": str(dataset_path),
                "results": results,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("Saved:", out_path)

print("\nAll datasets processed.")


Processing: SUI_single_hop_single_hop_negative_1.json
{"id": "SUI_A1", "type": "assessment", "is_starting": true, "text": "Has the person attempted a medically serious act of self-harm? (Check for signs of poisoning/intoxication, bleeding from self-inflicted wound, loss of consciousness, or extreme lethargy)", "user_answer": "NO"}
{"id": "SUI_A2", "type": "assessment", "is_starting": false, "text": "Is there an imminent risk of self-harm/suicide? (Current thoughts/plan of self-harm/suicide, or history of thoughts/plan in the past month or act in the past year in a person who is now extremely agitated, violent, distressed or lacks communication)", "user_answer": "NO"}
{"id": "SUI_A3", "type": "assessment", "is_starting": false, "text": "Is there a history of thoughts or plan of self-harm in the past month or act of self-harm in the past year?", "user_answer": "NO"}
{"id": "SUI_D3", "type": "diagnosis", "is_starting": false, "text": "Risk of self-harm/suicide is unlikely."}
{"id": "SUI_